In [17]:
import glob, time
import numpy as np
import pandas as pd
from scipy.linalg import qr as scipy_qr

In [18]:
def _a(X, r, w=None, f=None):
    Xw = X @ np.diag(w) if w is not None else X
    if f:
        AF = Xw[:, f]
        fr = [i for i in range(X.shape[1]) if i not in f]
        AR = Xw[:, fr]
        Q, _ = np.linalg.qr(AF)
        _, _, p = scipy_qr(AR - Q @ (Q.T @ AR), pivoting=True)
        piv = f + [fr[i] for i in p]
    else:
        _, _, piv = scipy_qr(Xw, pivoting=True)
    return piv[:r]

def _b(X, frac=0.7):
    n = int(X.shape[0] * frac)
    return X[:n], X[n:]

def _c(Xtr, Xte, sel, k):
    s = sel[:k]
    S = Xtr[:, s]; T = Xte[:, s]
    sol = np.linalg.lstsq(S.T, T.T, rcond=None)[0]
    R = np.maximum(sol.T @ Xtr, 1e-10)
    rmse = np.sqrt(np.mean((Xte - R) ** 2, axis=0))
    re = np.linalg.norm(R - Xte, 'fro') / np.linalg.norm(Xte, 'fro')
    return R, rmse, re

def _d(yt, yp):
    sr = np.sum((yt - yp) ** 2, axis=0)
    st = np.sum((yt - np.mean(yt, axis=0)) ** 2, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        n = np.where(st != 0, 1 - sr / st, np.nan)
    return n, 1 / (2 - n)

In [19]:
def _A(X, k, w=None, f=None):
    if hasattr(X, 'values'): X = X.values
    if w is not None:
        Xw = X * np.asarray(w)[np.newaxis, :]
    else:
        Xw = X
    Xw = np.asarray(Xw, dtype=np.float64, order='F')
    if f:
        f = list(f)
        AF = Xw[:, f]
        fr = [i for i in range(X.shape[1]) if i not in f]
        AR = Xw[:, fr]
        Q, _ = np.linalg.qr(AF)
        _, p = scipy_qr(AR - Q @ (Q.T @ AR), pivoting=True, mode='r')
        piv = np.array(f + [fr[i] for i in p])
    else:
        _, piv = scipy_qr(Xw, pivoting=True, mode='r')
        piv = np.array(piv)
    return piv[:k]

def _C(Xtr, Xte, sel, k):
    s = sel[:k]
    S = Xtr[:, s]; T = Xte[:, s]
    G = S.T @ S
    Y, *_ = np.linalg.lstsq(G, T.T, rcond=None)
    R = np.maximum(Y.T @ S.T @ Xtr, 1e-10)
    sq = (Xte - R) ** 2
    rmse = np.sqrt(np.mean(sq, axis=0))
    re = np.linalg.norm(R - Xte, 'fro') / np.linalg.norm(Xte, 'fro')
    sr = np.sum(sq, axis=0)
    st = np.sum((Xte - np.mean(Xte, axis=0)) ** 2, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        n = np.where(st != 0, 1 - sr / st, np.nan)
        nn = np.where((2 - n) != 0, 1 / (2 - n), np.nan)
    return dict(R=R, rmse=rmse, re=re, n=n, nn=nn)

def _E(Xtr, Xte, sel, k):
    s = sel[:k]
    S = Xtr[:, s]; T = Xte[:, s]
    Y = np.linalg.solve(S.T @ S, T.T)
    R = np.maximum(Y.T @ S.T @ Xtr, 1e-10)
    re = np.linalg.norm(R - Xte, 'fro') / np.linalg.norm(Xte, 'fro')
    return R, re

In [20]:
files = sorted(glob.glob('data/texas_gulf/streamflow_*.parquet'))
files = [f for f in files if '_clean' not in f]
t0 = time.time()
df = pd.concat([pd.read_parquet(f) for f in files], axis=0).sort_index()
data = df.values.astype(np.float64)
print(df.shape, f'{time.time()-t0:.2f}s')

Xtr, Xte = _b(data, 0.7)
print(Xtr.shape, Xte.shape)

(16040, 64954) 269.79s
(11228, 64954) (4812, 64954)


In [21]:
u = pd.read_csv('data/texas_gulf/usgs_sensors_texas_gulf.csv')
cset = set(u['comid'].astype(str))
cols = df.columns.astype(str)
k = len([c for c in cols if c in cset])
print(k)

447


In [22]:
t0 = time.time()
p1 = np.asarray(_a(Xtr, k))
print(f'{time.time()-t0:.2f}s')
t0 = time.time()
p2 = np.asarray(_A(Xtr, k))
print(f'{time.time()-t0:.2f}s')
print(np.array_equal(p1, p2), set(p1.tolist()) == set(p2.tolist()))

982.17s
1023.30s
True True


In [23]:
R1, rm1, re1 = _c(Xtr, Xte, p1, k)
r2 = _C(Xtr, Xte, p1, k)
print(re1, r2['re'], abs(re1 - r2['re']))
print(np.abs(R1 - r2['R']).max())
n1, nn1 = _d(Xte, R1)
print(np.nanmedian(n1), np.nanmedian(r2['n']))
print(np.nanmedian(nn1), np.nanmedian(r2['nn']))

0.032727838736290586 0.03272783873628816 2.42861286636753e-15
1.305961632169783e-07
0.490663631885902 0.4906636318883074
0.6625428374521253 0.6625428374531812


In [24]:
R3, re3 = _E(Xtr, Xte, p1, k)
print(re3, re1, abs(re3 - re1))
print(np.abs(R3 - R1).max())

0.03272783873628728 0.032727838736290586 3.3029134982598407e-15
3.161221684422344e-08
